# Quadratic portfolio construction: PGD versus exact and standard solvers

This notebook implements the quadratic model

$$
\min_h\;\frac{\lambda}{2}h^\top Vh-\alpha^\top h
+\frac{\theta}{2}(h-h_-)^\top Q(h-h_-)
\quad\text{subject to}\quad C^\top h=c.
$$

We compare three independent solution routes:

1. projected gradient descent with exact affine projection;
2. the exact equality-constrained KKT system;
3. SciPy's standard SLSQP solver.

The numerical assertions at the end turn the notebook into an executable validation document.

## PGD iteration

With negative utility denoted by $F(h)$,

$$
\nabla F(h)=\lambda Vh+\theta Q(h-h_-)-\alpha.
$$

The Euclidean affine projection is

$$
\Pi(z)=z-C(C^\top C)^{\dagger}(C^\top z-c).
$$

The implementation uses a projected-majorization line search, records the projected-gradient norm,
and refuses to return a portfolio whose constraints exceed the requested tolerance.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.8g}")

In [ ]:
from portfolio_pgd import (
    ConstraintSet,
    PGDOptions,
    PortfolioProblem,
    factor_covariance,
    solve_pgd,
    solve_quadratic_kkt,
    solve_scipy_slsqp,
)

n_assets = 36
rng = np.random.default_rng(1201)
covariance, loadings = factor_covariance(n_assets, 4, seed=1202, specific_risk=0.12)
alpha = rng.normal(scale=0.025, size=n_assets)
previous = rng.normal(scale=0.01, size=n_assets)
q_diagonal = 0.4 + rng.random(n_assets)

problem = PortfolioProblem(
    alpha=alpha,
    covariance=covariance,
    previous_holdings=previous,
    risk_aversion=2.25,
    quadratic_cost_matrix=q_diagonal,
    quadratic_cost_aversion=0.75,
)

# Full investment and one factor-exposure target.
factor_direction = loadings[:, 0] - np.mean(loadings[:, 0])
C_transpose = np.vstack([np.ones(n_assets), factor_direction])
targets = np.array([1.0, 0.0])
constraints = ConstraintSet(
    n_assets,
    equality_matrix=C_transpose,
    equality_target=targets,
)
print("Minimum covariance eigenvalue:", np.linalg.eigvalsh(covariance).min())

In [ ]:
pgd = solve_pgd(
    problem,
    constraints,
    options=PGDOptions(max_iterations=25_000, tolerance=2e-9),
)
kkt = solve_quadratic_kkt(problem, constraints)
slsqp = solve_scipy_slsqp(problem, constraints)

comparison = pd.DataFrame(
    {
        "objective": [pgd.objective, kkt.objective, slsqp.objective],
        "utility": [pgd.utility, -kkt.objective, -slsqp.objective],
        "distance_to_KKT": [
            np.linalg.norm(pgd.holdings - kkt.holdings),
            0.0,
            np.linalg.norm(slsqp.holdings - kkt.holdings),
        ],
        "max_constraint_violation": [
            constraints.max_violation(pgd.holdings),
            constraints.max_violation(kkt.holdings),
            constraints.max_violation(slsqp.holdings),
        ],
    },
    index=["PGD", "Exact KKT", "SciPy SLSQP"],
)
print(comparison.to_string())
print(f"\nPGD status={pgd.status}; iterations={pgd.iterations}; SLSQP={slsqp.message}")

## Convergence audit

For a convex quadratic, the KKT objective is the global minimum of the estimated problem. The first
panel plots the PGD objective gap; the second shows the norm of the projected-gradient mapping.

In [ ]:
history = pd.DataFrame(pgd.history)
objective_gap = np.maximum(history["objective"] - kkt.objective, 1e-18)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(history["iteration"], objective_gap)
axes[0].set(title="Objective gap to exact KKT", xlabel="Iteration", ylabel="F(h) - F(h*)")
valid = history["projected_gradient_norm"].notna()
axes[1].semilogy(
    history.loc[valid, "iteration"],
    np.maximum(history.loc[valid, "projected_gradient_norm"], 1e-18),
)
axes[1].set(title="First-order residual", xlabel="Iteration", ylabel="Projected-gradient norm")
plt.tight_layout()
plt.show()

## Alpha, persistence, and constraint decomposition

Let $H=\lambda V+\theta Q$ and $b=\alpha+\theta Qh_-$. The unconstrained target is $H^{-1}b$.
The exact constrained solution decomposes as

$$
h^*=\underbrace{H^{-1}\alpha}_{\text{alpha}}
+\underbrace{H^{-1}\theta Qh_-}_{\text{persistence}}
-\underbrace{H^{-1}C\mu}_{\text{constraint correction}}.
$$

In [ ]:
H = problem.quadratic_hessian
Q = np.asarray(problem.quadratic_cost_matrix)
A = np.asarray(constraints.equality_matrix)
c = np.asarray(constraints.equality_target)

alpha_component = np.linalg.solve(H, alpha)
persistence_component = np.linalg.solve(
    H, problem.quadratic_cost_aversion * Q @ previous
)
unconstrained = alpha_component + persistence_component
H_inv_A_T = np.linalg.solve(H, A.T)
mu = np.linalg.solve(A @ H_inv_A_T, A @ unconstrained - c)
constraint_component = -H_inv_A_T @ mu
reconstructed = alpha_component + persistence_component + constraint_component

decomposition = pd.DataFrame(
    {
        "L2 norm": [
            np.linalg.norm(alpha_component),
            np.linalg.norm(persistence_component),
            np.linalg.norm(constraint_component),
            np.linalg.norm(reconstructed),
        ],
        "net exposure": [
            np.sum(alpha_component),
            np.sum(persistence_component),
            np.sum(constraint_component),
            np.sum(reconstructed),
        ],
    },
    index=["alpha", "persistence", "constraint correction", "total"],
)
print(decomposition.to_string())
print("Reconstruction error:", np.linalg.norm(reconstructed - kkt.holdings))

## Executable acceptance tests

In [ ]:
assert pgd.converged
assert slsqp.success
assert constraints.max_violation(pgd.holdings) < 1e-8
assert np.linalg.norm(pgd.holdings - kkt.holdings) < 5e-7
assert np.linalg.norm(slsqp.holdings - kkt.holdings) < 5e-6
assert np.linalg.norm(reconstructed - kkt.holdings) < 1e-10
print("All quadratic validation checks passed.")